In [1]:
!pip install matplotlib seaborn plotly nltk scikit-learn wordcloud


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import nltk
from wordcloud import WordCloud
import sklearn
import re 
from nltk.tokenize import word_tokenize  
from nltk.stem import WordNetLemmatizer  
from nltk.corpus import stopwords
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\prama\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
print('Pandas Version:', pd.__version__)
print('Numpy Version:', np.__version__)
print('Matplotlib Version:', plt.matplotlib.__version__)
print('Seaborn Version:', sns.__version__)
print("All libraries loaded successfully")

Pandas Version: 2.3.3
Numpy Version: 2.2.6
Matplotlib Version: 3.10.8
Seaborn Version: 0.13.2
All libraries loaded successfully


# Text Cleaning Pipeline

## What is Text Cleaning?
Text cleaning is the process of removing unwanted symbols, 
noise, and irrelevant text from raw data to prepare it for 
NLP analysis. Without cleaning, models learn from noise 
instead of meaningful content.

## Noise Found in Raw Tweets
After examining the raw dataset we identified the following:
- @mentions (e.g. @united, @AmericanAir)
- URLs (e.g. http://t.co/xyz)
- Hashtag symbols (e.g. #neveragain)
- Numbers and special characters
- Extra whitespace

## Data Preprocessing Steps
1. Removed duplicate rows from the dataset
2. Checked missing values across all 15 columns
3. Dropped 5 irrelevant columns (tweet_coord, tweet_location, 
   airline_sentiment_gold, negativereason_gold, 
   negativereason_confidence)
4. Filtered rows with airline_sentiment_confidence below 0.65
   to remove uncertain labels — retained 93.4% of data

## Text Cleaning Steps
1. **Lowercase** — standardizes all text
2. **Remove URLs** — carry no sentiment meaning
3. **Remove @mentions** — airline already captured separately
4. **Remove hashtag symbol, keep word** — words carry sentiment
5. **Remove numbers** — sentiment is in words not numbers
6. **Tokenization** — breaks sentences into individual words
7. **Stop words removal** — removes meaningless words 
   but keeps: no, not, never, without, nor
8. **Lemmatization** — reduces words to base form 
   (flights → flight, waited → wait)
9. **Filter short tweets** — removed tweets with less than 
   3 words after cleaning

## Result
Input:  '@united horrible flight!! waited 3 hours http://t.co/xyz'
Output: 'horrible flight waited hour'

## Final Dataset
- Started with: 14,640 rows
- Final clean dataset: 12,679 rows × 10 columns
- 

In [5]:
#Loading the dataset
df = pd.read_csv("../data/raw/tweets.csv", encoding='latin-1')

In [6]:
df

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14635,569587686496825344,positive,0.3487,NaN,0.0000,American,NaN,KristenReenders,NaN,0,@AmericanAir thank you we got on a different f...,NaN,2015-02-22 12:01:01 -0800,NaN,NaN
14636,569587371693355008,negative,1.0000,Customer Service Issue,1.0000,American,NaN,itsropes,NaN,0,@AmericanAir leaving over 20 minutes Late Flig...,NaN,2015-02-22 11:59:46 -0800,Texas,NaN
14637,569587242672398336,neutral,1.0000,NaN,NaN,American,NaN,sanyabun,NaN,0,@AmericanAir Please bring American Airlines to...,NaN,2015-02-22 11:59:15 -0800,"Nigeria,lagos",NaN
14638,569587188687634433,negative,1.0000,Customer Service Issue,0.6659,American,NaN,SraJackson,NaN,0,"@AmericanAir you have my money, you change my ...",NaN,2015-02-22 11:59:02 -0800,New Jersey,Eastern Time (US & Canada)


In [7]:
duplicated_rows = df.duplicated()
print(" How many rows where duplicated?", duplicated_rows.sum())

 How many rows where duplicated? 36


In [8]:
df.drop_duplicates()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14635,569587686496825344,positive,0.3487,NaN,0.0000,American,NaN,KristenReenders,NaN,0,@AmericanAir thank you we got on a different f...,NaN,2015-02-22 12:01:01 -0800,NaN,NaN
14636,569587371693355008,negative,1.0000,Customer Service Issue,1.0000,American,NaN,itsropes,NaN,0,@AmericanAir leaving over 20 minutes Late Flig...,NaN,2015-02-22 11:59:46 -0800,Texas,NaN
14637,569587242672398336,neutral,1.0000,NaN,NaN,American,NaN,sanyabun,NaN,0,@AmericanAir Please bring American Airlines to...,NaN,2015-02-22 11:59:15 -0800,"Nigeria,lagos",NaN
14638,569587188687634433,negative,1.0000,Customer Service Issue,0.6659,American,NaN,SraJackson,NaN,0,"@AmericanAir you have my money, you change my ...",NaN,2015-02-22 11:59:02 -0800,New Jersey,Eastern Time (US & Canada)


In [9]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created                 14640 non-null  object 
 13  t

In [10]:
#Handling missing values
print(df.isnull().sum())

tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64


Removed 3 columns with 14600 missing values because removing the nulls would give us 30-40 rows to do analysis which is highly impossible.

In [11]:
df.drop(columns=["airline_sentiment_gold", "negativereason_gold", "tweet_coord", "tweet_location"], inplace=True)

In [12]:
df.shape

(14640, 11)

In [13]:
df['airline_sentiment_confidence'].describe()

count    14640.000000
mean         0.900169
std          0.162830
min          0.335000
25%          0.692300
50%          1.000000
75%          1.000000
max          1.000000
Name: airline_sentiment_confidence, dtype: float64

Imbalance Data
Counts how many tweets are positive, negative, neutral. Stores it in a variable called counts so you can reuse it.

In [14]:
print(df['airline'].value_counts())

airline
United            3822
US Airways        2913
American          2759
Southwest         2420
Delta             2222
Virgin America     504
Name: count, dtype: int64


In [15]:
print('\n=== SENTIMENT BREAKDOWN ===')
counts = df['airline_sentiment'].value_counts()
pcts   = df['airline_sentiment'].value_counts(normalize=True).mul(100).round(1)
for sentiment in counts.index:
    print(f'  {sentiment:10s}: {counts[sentiment]:5,} tweets  ({pcts[sentiment]}%)')



=== SENTIMENT BREAKDOWN ===
  negative  : 9,178 tweets  (62.7%)
  neutral   : 3,099 tweets  (21.2%)
  positive  : 2,363 tweets  (16.1%)


In [16]:
lemmatizer = WordNetLemmatizer()  
stop_words = set(stopwords.words('english'))  
stop_words -= {'no', 'not', 'never', 'without', 'nor'} 

def clean_tweets(text):     
    if not isinstance(text,str):         
        return '' 
    text = text.lower() 
    text = re.sub(r'http\S+', '', text)  
    text = re.sub(r'@\w+', '', text)  
    text = re.sub(r'#(\w+)', r'\1', text)  
    text = re.sub(r'\d+', '', text)  
    text = re.sub(r'[^a-z\s]', '', text) 
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    tokens = word_tokenize(text) 
    tokens = [word for word in tokens if word not in stop_words and len(word)>2 ]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  
    return ' '.join(tokens)

In [17]:
df['text_clean'] = df['text'].apply(clean_tweets)
print(df.shape)
df[['text', 'text_clean']].head(20)

(14640, 12)


,text,text_clean
0,@VirginAmerica What @dhepburn said.,said
1,@VirginAmerica plus you've added commercials t...,plus youve added commercial experience tacky
2,@VirginAmerica I didn't today... Must mean I n...,didnt today must mean need take another trip
3,@VirginAmerica it's really aggressive to blast...,really aggressive blast obnoxious entertainmen...
4,@VirginAmerica and it's a really big bad thing...,really big bad thing
5,@VirginAmerica seriously would pay $30 a fligh...,seriously would pay flight seat didnt playing ...
6,"@VirginAmerica yes, nearly every time I fly VX...",yes nearly every time fly ear worm wont away
7,@VirginAmerica Really missed a prime opportuni...,really missed prime opportunity men without ha...
8,"@virginamerica Well, I didn'tâ¦but NOW I DO! :-D",well didntbut
9,"@VirginAmerica it was amazing, and arrived an ...",amazing arrived hour early youre good


In [18]:
df = df[df['text_clean'].str.split().str.len() > 2]
print(df.shape)
df.reset_index(drop=True, inplace=True)

(13574, 12)


In [19]:
print(df.columns)

Index(['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence',
       'negativereason', 'negativereason_confidence', 'airline', 'name',
       'retweet_count', 'text', 'tweet_created', 'user_timezone',
       'text_clean'],
      dtype='object')


In [20]:
print(df['airline_sentiment_confidence'].describe())


count    13574.000000
mean         0.904631
std          0.160776
min          0.335000
25%          0.698700
50%          1.000000
75%          1.000000
max          1.000000
Name: airline_sentiment_confidence, dtype: float64


In [21]:
# How many rows have confidence <= 0.65?
low_conf = df[df['airline_sentiment_confidence'] <= 0.65]
print(f'Rows below 0.65: {len(low_conf)}')
print(f'Percentage lost: {len(low_conf)/len(df)*100:.1f}%')

# How many rows have confidence <= 0.70?
low_conf70 = df[df['airline_sentiment_confidence'] <= 0.66]
print(f'Rows below 0.66: {len(low_conf70)}')
print(f'Percentage lost: {len(low_conf70)/len(df)*100:.1f}%')

Rows below 0.65: 895
Percentage lost: 6.6%
Rows below 0.66: 1485
Percentage lost: 10.9%


In [22]:
df=df[df['airline_sentiment_confidence']>0.65]
df.drop(columns=['airline_sentiment_confidence' , 'negativereason_confidence'],inplace=True)
print(df.shape)
print(df.columns)

(12679, 10)
Index(['tweet_id', 'airline_sentiment', 'negativereason', 'airline', 'name',
       'retweet_count', 'text', 'tweet_created', 'user_timezone',
       'text_clean'],
      dtype='object')


In [23]:
df.to_csv("../data/processed/tweets_clean.csv", index = False)
print('Saved')

Saved
